# Visualisierung der Singularitäten beim vereinfachten Scara Roboter

Jonas Frei, 21.09.2026, jonas.frei@ost.ch

In [ ]:
import numpy as np
import plotly.graph_objects as go
from matplotlib import pyplot as plt

## Parameter definition

Längen der beiden Roboterglieder

In [ ]:
l1 = 1
l2 = 1

Jacobi-Matrix

In [ ]:
def J(q1, q2):
    return np.array([
        [-l1 * np.sin(q1) - l2 * np.sin(q1 + q2), -l2 * np.sin(q1 + q2)],
        [ l1 * np.cos(q1) + l2 * np.cos(q1 + q2),  l2 * np.cos(q1 + q2)]
    ])

## Einige Beispiele

In [ ]:
print(f"q1 = 0, q2 = 0\nJ = {J(0, 0)}\n|J| = {np.linalg.det(J(0, 0))}\n")

print(f"q1 = 0, q2 = pi/2\nJ = {J(0, np.pi/2)}\n|J| = {np.linalg.det(J(0, np.pi/2))}\n")

print(f"q1 = pi/2, q2 = 0\nJ = {J(np.pi/2, 0)}\n|J| = {np.linalg.det(J(np.pi/2, 0))}\n")

print(f"q1 = pi/2, q2 = pi/2\nJ = {J(np.pi/2, np.pi/2)}\n|J| = {np.linalg.det(J(np.pi/2, np.pi/2))}\n")

print(f"q1 = 0, q2 = pi\nJ = {J(0, np.pi)}\n|J| = {np.linalg.det(J(0, np.pi))}\n")

## Visualisierung

Anlegen eines Meshgrids

In [ ]:
# MATLAB: [q1,q2] = meshgrid(-pi:pi/10:pi, -pi:pi/20:2*pi)
q1_vals = np.linspace(-np.pi, np.pi, 21)        # Schrittweite pi/10
q2_vals = np.linspace(-2*np.pi, 2 * np.pi, 81)    # Schrittweite pi/20
q1, q2 = np.meshgrid(q1_vals, q2_vals)

Berechnen der Determinante der Jacobi-Matrix

In [ ]:
detJ = np.round(
    (-l1 * np.sin(q1) - l2 * np.sin(q1 + q2)) * (l2 * np.cos(q1 + q2))
    - (l1 * np.cos(q1) + l2 * np.cos(q1 + q2)) * (-l2 * np.sin(q1 + q2)),
    5
)

Plotten: Die Nullstellen-Kontur (rote Linie) entspricht den Singularitäten.

In [ ]:
# Nullstellen-Kontur (Singularitäten) berechnen, ohne sie mit matplotlib anzuzeigen
_fig_tmp, _ax_tmp = plt.subplots()
_cs = _ax_tmp.contour(q1, q2, detJ, levels=[0])
_segments = _cs.allsegs[0]
plt.close(_fig_tmp)

contour_traces = [
    go.Scatter3d(
        x=seg[:, 0], y=seg[:, 1], z=np.zeros(len(seg)),
        mode='lines', line=dict(color='red', width=6),
        name='|J| = 0', showlegend=False
    )
    for i, seg in enumerate(_segments)
]

surface = go.Surface(
    x=q1, y=q2, z=detJ,
    colorscale='Viridis',
    opacity=0.85,
    colorbar=dict(title='|J|')
)

# Konturlinien zuerst hinzufügen, Fläche (transparent) danach
fig = go.Figure(data=contour_traces + [surface])

pi_ticks = np.arange(-np.pi, np.pi + 1e-9, np.pi / 4)
pi_labels = ['-π', '-3π/4', '-π/2', '-π/4', '0', 'π/4', 'π/2', '3π/4', 'π']

fig.update_layout(
    title='Determinante der Jakobimatrix des vereinfachten Scara Roboters',
    scene=dict(
        xaxis=dict(title='q1', range=[-np.pi, np.pi], tickvals=pi_ticks, ticktext=pi_labels),
        yaxis=dict(title='q2', range=[-np.pi, np.pi], tickvals=pi_ticks, ticktext=pi_labels),
        zaxis=dict(title='|J|')
    ),
    width=850,
    height=700,
    margin=dict(l=0, r=0, t=60, b=0)
)

fig.show()

Die Determinante der Jacobi-Matrix wird $0$, wenn $q_2 = n\cdot \pi,\;\;\;\;\;n\in \mathbb{Z}\;$